# Empirical standardized-return CDF — public calibration screen

## tl;dr

- **Reject `empirical_standardized_return_cdf_v1` without retuning.** Across 10,062 complete BTC five-minute conditions and 30,186 fixed-offset forecasts, aggregate Brier improvement is `+0.000521`, but log-loss improvement is only `+0.000302` versus the preregistered `+0.001000` floor.
- The mechanism fails eight fixed gates: both bootstrap lower bounds, both scores in the first older subwindow, 120-second log loss, both overconfidence-tail scores, and the aggregate log-loss magnitude.
- The candidate moves `93.84%` of forecasts farther from 0.5 and worsens the high-confidence tail, contradicting its intended role as an uncertainty correction.
- **No strategy variant, exact replay, runtime change, paper/live trading, profitability claim, or A+ credit is authorized.** The active binary-complement forward block remains sealed.

## Context & Methods

The current engine maps a causal one-hour realized-volatility estimate into a Gaussian/lognormal binary terminal probability. The registered candidate keeps the same spot, strike, volatility scale, 0.30 floor, and decision offsets, but replaces the standard-normal residual law with the empirical distribution of strictly prior same-offset five-minute standardized returns.

### Key Assumptions

- Each forecast may use only fully completed prior conditions from the trailing seven days.
- Residual histories are separated at 120, 150, and 179 seconds so horizons are never mixed.
- The candidate uses fixed Jeffreys half-success/half-failure smoothing and requires at least 1,800 prior samples; no lookback, smoothing, or threshold neighbors are tested.
- April 16–May 15 is the older evaluation population. July 16–20 is the freshest fully resolved holdout. Strict-42, earlier proxy screens, retained July 14–15 captures, and all active or sealed forward conditions are excluded from evaluation.
- Binance terminal direction is a public proxy, not official Chainlink settlement or executable Polymarket PnL.

In [1]:
from pathlib import Path
import gzip
import hashlib
import json
import subprocess
import sys

import pandas as pd

ROOT = Path('/Users/ttoomm/Documents/PolyMomentum')
REGISTRY = ROOT / 'deploy/promotions/evidence/strategy_registry'
ARCHIVE_DIR = Path('/private/tmp/polymomentum-dvol-volatility-max-20260721/binance_1s')
PREREGISTRATION = REGISTRY / '20260721_empirical_return_cdf_preregistration.json'
EVIDENCE = REGISTRY / '20260721_empirical_return_cdf_public_calibration.json'
SNAPSHOT = REGISTRY / 'source_snapshots/20260721_empirical_return_cdf_forecasts.jsonl.gz'

def sha256_file(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

analysis = subprocess.run(
    [
        sys.executable,
        str(ROOT / 'scripts/analyze_empirical_return_cdf.py'),
        '--archive-dir', str(ARCHIVE_DIR),
        '--evidence', str(EVIDENCE),
        '--snapshot', str(SNAPSHOT),
    ],
    check=True,
    capture_output=True,
    text=True,
)
analysis_summary = json.loads(analysis.stdout)
preregistration = json.loads(PREREGISTRATION.read_text())
evidence = json.loads(EVIDENCE.read_text())
{
    'analysis_status': analysis_summary['status'],
    'registration_status': preregistration['status'],
    'result_status': evidence['status'],
    'archives': evidence['source_data_quality']['archives'],
    'source_rows': evidence['source_data_quality']['rows'],
    'conditions': evidence['results']['overall']['conditions'],
    'forecasts': evidence['results']['overall']['forecasts'],
}

{'analysis_status': 'PUBLIC_PROXY_CALIBRATION_REJECTED_NO_RETUNING',
 'registration_status': 'PREREGISTERED_BEFORE_NEW_EVALUATION_LABEL_DOWNLOAD',
 'result_status': 'PUBLIC_PROXY_CALIBRATION_REJECTED_NO_RETUNING',
 'archives': 49,
 'source_rows': 4233600,
 'conditions': 10062,
 'forecasts': 30186}

## Data

The source population is 49 official Binance BTCUSDT one-second daily archives across two disjoint segments. Every adjacent checksum, 12-column schema, timestamp, one-second interval, close duration, and positive price is validated before forecasting. The expected April-to-July gap is explicit and never enters a rolling volatility window.

In [2]:
for source in preregistration['source_pins_before_evaluation'].values():
    assert sha256_file(ROOT / source['path']) == source['sha256']
authority = evidence['authority']
assert sha256_file(PREREGISTRATION) == authority['preregistration_sha256']
compressed_snapshot = SNAPSHOT.read_bytes()
uncompressed_snapshot = gzip.decompress(compressed_snapshot)
assert hashlib.sha256(compressed_snapshot).hexdigest() == authority['forecast_snapshot']['sha256']
assert hashlib.sha256(uncompressed_snapshot).hexdigest() == authority['forecast_snapshot']['uncompressed_sha256']
assert len(uncompressed_snapshot.splitlines()) == authority['forecast_snapshot']['rows']

source_quality = evidence['source_data_quality']
forecast_quality = evidence['forecast_data_quality']
pd.DataFrame([
    {'check': 'Official archives', 'observed': source_quality['archives'], 'required': 49},
    {'check': 'One-second source rows', 'observed': source_quality['rows'], 'required': 4_233_600},
    {'check': 'Checksum failures', 'observed': source_quality['checksum_failures'], 'required': 0},
    {'check': 'Internal gap violations', 'observed': source_quality['one_second_gap_violations_within_segments'], 'required': 0},
    {'check': 'Duplicate condition-offsets', 'observed': forecast_quality['duplicate_condition_offsets'], 'required': 0},
    {'check': 'Minimum prior samples', 'observed': forecast_quality['prior_sample_count_minimum'], 'required': 1_800},
    {'check': 'Forecast completeness', 'observed': forecast_quality['complete_registered_forecasts_fraction'], 'required': 0.99},
])

,check,observed,required
0,Official archives,4.900000e+01,49.00
1,One-second source rows,4.233600e+06,4233600.00
2,Checksum failures,0.000000e+00,0.00
3,Internal gap violations,0.000000e+00,0.00
4,Duplicate condition-offsets,0.000000e+00,0.00
5,Minimum prior samples,2.016000e+03,1800.00
6,Forecast completeness,9.982143e-01,0.99


## Results

The aggregate Brier gain narrowly clears its magnitude floor, but the log-loss gain does not. More importantly, performance is unstable: the first older subwindow is negative on both scores, the 120-second log-loss delta is slightly negative, the high-confidence tail deteriorates, and both paired day-bootstrap intervals cross zero. A compact table is more honest than a chart because the decision is conjunctive across exact gates rather than driven by a visual trend.

In [3]:
def score_row(scope, score):
    return {
        'scope': scope,
        'conditions': score['conditions'],
        'forecasts': score['forecasts'],
        'brier_improvement': score['brier_improvement'],
        'log_loss_improvement': score['log_loss_improvement'],
        'mean_abs_probability_displacement': score['mean_absolute_probability_displacement'],
        'moved_away_from_half_fraction': score['moved_away_from_half_fraction'],
    }

result_rows = [score_row('overall', evidence['results']['overall'])]
result_rows.extend(
    score_row(name, score)
    for name, score in evidence['results']['chronological_windows'].items()
)
result_rows.extend(
    score_row(f'offset_{name}s', score)
    for name, score in evidence['results']['decision_offsets'].items()
)
result_rows.append(score_row('overconfidence_tail', evidence['results']['overconfidence_tail']))
pd.DataFrame(result_rows)

,scope,conditions,forecasts,brier_improvement,log_loss_improvement,mean_abs_probability_displacement,moved_away_from_half_fraction
0,overall,10062,30186,0.000521,0.000302,0.044238,0.938382
1,fresh_holdout,1435,4305,0.000575,0.001628,0.047393,0.939837
2,older_first,4315,12945,-0.000069,-0.001545,0.033968,0.917188
3,older_second,4312,12936,0.001092,0.001708,0.053466,0.959106
4,offset_120s,10062,10062,0.000491,-0.000010,0.045489,0.959948
5,offset_150s,10062,10062,0.000588,0.000239,0.045208,0.939177
6,offset_179s,10062,10062,0.000483,0.000676,0.042017,0.916021
7,overconfidence_tail,5378,11581,-0.000261,-0.002589,0.038365,0.860806


In [4]:
bootstrap = evidence['results']['bootstrap']
failed_checks = evidence['gate_evaluation']['failed_checks']
assert evidence['status'] == 'PUBLIC_PROXY_CALIBRATION_REJECTED_NO_RETUNING'
assert evidence['gate_evaluation']['passed'] is False
assert evidence['decision']['strategy_variant_authorized'] is False
assert evidence['decision']['runtime_change_authorized'] is False
assert evidence['decision']['profitability_claim'] is False
assert evidence['decision']['a_plus_claim'] is False
{
    'brier_improvement_95pct': bootstrap['brier_improvement_95pct'],
    'log_loss_improvement_95pct': bootstrap['log_loss_improvement_95pct'],
    'failed_check_count': len(failed_checks),
    'failed_checks': failed_checks,
}

{'brier_improvement_95pct': [-0.0007733144700255535, 0.0018856320878563625],
 'log_loss_improvement_95pct': [-0.003228738009282189, 0.004003221684693689],
 'failed_check_count': 8,
 'failed_checks': ['overall_log_loss_improvement_at_least_0_001',
  'brier_bootstrap_lower_bound_positive',
  'log_loss_bootstrap_lower_bound_positive',
  'all_chronological_windows_improve_brier',
  'all_chronological_windows_improve_log_loss',
  'each_decision_offset_log_loss_nonnegative',
  'overconfidence_tail_brier_nonnegative',
  'overconfidence_tail_log_loss_nonnegative']}

## Independent Validation

The validator does not import the analysis script. It rechecks all 49 archive hashes and schemas, independently computes rolling variance with cumulative sums, reconstructs every historical residual and causal seven-day sample, reproduces every candidate and baseline probability, recomputes proper scores and the deterministic day bootstrap, and confirms the frozen rejection.

In [5]:
validation = subprocess.run(
    [sys.executable, str(ROOT / 'scripts/validate_empirical_return_cdf.py'), '--archive-dir', str(ARCHIVE_DIR)],
    check=True,
    capture_output=True,
    text=True,
)
validation_result = json.loads(validation.stdout)
assert validation_result['ok'] is True
validation_result

{'ok': True,
 'archives': 49,
 'source_rows': 4233600,
 'conditions': 10062,
 'forecasts': 30186,
 'terminal_ties': 18,
 'brier_improvement': 0.0005205290753615272,
 'log_loss_improvement': 0.00030154935441539864,
 'failed_checks': ['overall_log_loss_improvement_at_least_0_001',
  'brier_bootstrap_lower_bound_positive',
  'log_loss_bootstrap_lower_bound_positive',
  'all_chronological_windows_improve_brier',
  'all_chronological_windows_improve_log_loss',
  'each_decision_offset_log_loss_nonnegative',
  'overconfidence_tail_brier_nonnegative',
  'overconfidence_tail_log_loss_nonnegative']}

## Takeaways

1. **Reject the family as registered.** Do not tune the lookback, smoothing, residual standardization, dates, offsets, or gates on these windows.
2. **Do not implement it in replay or runtime.** Aggregate Brier improvement is insufficient when log loss, uncertainty, chronology, offset stability, and the overconfidence tail fail.
3. **Preserve the active decision order.** Finish the sealed binary-complement block at 750 conditions and score it once. The official settlement-anchor contract remains the only later model-specification candidate, with negative but underpowered retrospective evidence.
4. **No A+ or profitability claim follows.** This is a public-proxy model screen, not executable strategy evidence.